> Projeto Desenvolve <br>
Programação Intermediária com Python <br>
Profa. Camila Laranjeira (mila@projetodesenvolve.com.br) <br>

# 3.6 - Pandas

## Exercícios
Antes de continuar, baixe os arquivos das bases de dados de partidas em Copas do Mundo e salve na mesma pasta deste notebook.
* https://raw.githubusercontent.com/camilalaranjeira/python-intermediario/main/fifa-wc/matches_1930_2022.csv
* https://raw.githubusercontent.com/camilalaranjeira/python-intermediario/main/fifa-wc/matches_1991_2023.csv

A célula a seguir já carrega os dados em CSV e ajusta as colunas para trabalharmos com os nomes traduzidos (como fizemos em aula).

In [ ]:
import pandas as pd
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',12)

wcwomen_df = pd.read_csv('matches_1991_2023.csv')
wcmen_df   = pd.read_csv('matches_1930_2022.csv')
wc = pd.concat((wcwomen_df,wcmen_df)).reset_index()

nomes_traduzidos = {'home_team': 'time_1', 'away_team': 'time_2', 'home_score': 'gols_1', 'away_score': 'gols_2',
                    'Date': 'data', 'Year': 'ano', 'Host': 'país_sede', 'Attendance': 'comparecimento',
                    'Score': 'resultado', 'Round': 'rodada', 'home_goal': 'gols_1_detalhes', 'away_goal': 'gols_2_detalhes',
                    'home_own_goal': 'gols_1_contra', 'away_own_goal': 'gols_2_contra',
                    'home_penalty_goal': 'gols_1_penalti', 'away_penalty_goal': 'gols_2_penalti',
                    'home_red_card': 'cartao_vermelho_1', 'away_red_card': 'cartao_vermelho_2',
                    'home_yellow_card_long': 'cartao_amarelo_1', 'away_yellow_card_long': 'cartao_amarelo_2'}

wc = wc.loc[:, nomes_traduzidos.keys()]
wc.columns = nomes_traduzidos.values()

copa = wc['ano'].apply( lambda x: 'Masculina' if x % 2 == 0 else 'Feminina').astype('string')
wc['copa'] = copa
display(wc)

#### Q1.
Faça as conversões de tipo necessárias para que a saída do comando `wc.info()` seja como apresentado a seguir. E **salve o novo dataframe** com o comando `df.to_csv('wc_formatado.csv')`.

```
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   time_1             1312 non-null   string        
 1   time_2             1312 non-null   string        
 2   gols_1             1312 non-null   int64         
 3   gols_2             1312 non-null   int64         
 4   data               1312 non-null   datetime64[ns]
 5   ano                1312 non-null   int64         
 6   país_sede          1312 non-null   string        
 7   comparecimento     1312 non-null   int64         
 8   resultado          1312 non-null   string        
 9   rodada             1312 non-null   category      
 10  gols_1_detalhes    970 non-null    string        
 11  gols_2_detalhes    771 non-null    string        
 12  gols_1_contra      57 non-null     string        
 13  gols_2_contra      30 non-null     string        
 14  gols_1_penalti     170 non-null    string        
 15  gols_2_penalti     119 non-null    string        
 16  cartao_vermelho_1  59 non-null     string        
 17  cartao_vermelho_2  65 non-null     string        
 18  cartao_amarelo_1   834 non-null    string        
 19  cartao_amarelo_2   857 non-null    string        
 20  copa               1312 non-null   string 
```

In [ ]:
# 1. Conversões de tipo
cols_string = ['time_1', 'time_2', 'país_sede', 'resultado', 'gols_1_detalhes', 
               'gols_2_detalhes', 'gols_1_contra', 'gols_2_contra', 'gols_1_penalti', 
               'gols_2_penalti', 'cartao_vermelho_1', 'cartao_vermelho_2', 
               'cartao_amarelo_1', 'cartao_amarelo_2', 'copa']

for col in cols_string:
    wc[col] = wc[col].astype('string')

wc['data'] = pd.to_datetime(wc['data'])
wc['rodada'] = wc['rodada'].astype('category')

# Tratando comparecimento (NaN para 0 antes de converter para int)
wc['comparecimento'] = wc['comparecimento'].fillna(0).astype('int64')
wc['gols_1'] = wc['gols_1'].astype('int64')
wc['gols_2'] = wc['gols_2'].astype('int64')
wc['ano'] = wc['ano'].astype('int64')

# Salvando
wc.to_csv('wc_formatado.csv', index=False)
wc.info()

#### Q2.
Apresente a linha do dataframe `wc` que corresponde ao jogo com maior audiência da história.

In [ ]:
jogo_recorde = wc.loc[[wc['comparecimento'].idxmax()]]
display(jogo_recorde)

#### Q3.
Aplicando operações sobre as colunas `ano` e `copa` do dataframe `wc`, apresente quantas copas femininas e quantas copas masculinas já aconteceram na história.

Exemplo de saída (valores inventados):
```
Masculina: 22
Feminina: 9
```

In [ ]:
contagem = wc.groupby('copa')['ano'].nunique()
print(f"Masculina: {contagem['Masculina']}")
print(f"Feminina: {contagem['Feminina']}")

#### Q3. 
Crie um dataframe `participacao` com as colunas `país`, `copa`, e `num_copas` que armazena a quantidade de copas do mundo que cada país participou, informando se é da copa masculina ou feminina. Ao final imprima, como apresentado a seguir, o top 5 países de cada competição que mais participou de copas do mundo.

Exemplo de saída (valores inventados):
```
país            copa        num_copas
Brazil          Feminina    8
Unites States   Feminina    8
Germany         Feminina    8
Japan           Feminina    7
Colombia        Feminina    7
```

```
país            copa        num_copas
Brazil          Masculina   21
Germany         Masculina   21
Argentina       Masculina   20
England         Masculina   20
Mexico          Masculina   20
```

In [ ]:
# Criamos uma lista de todos os países que jogaram (em casa ou fora) em cada ano/copa
part_1 = wc[['time_1', 'ano', 'copa']].rename(columns={'time_1': 'país'})
part_2 = wc[['time_2', 'ano', 'copa']].rename(columns={'time_2': 'país'})
todas_part = pd.concat([part_1, part_2])

# Contamos quantas edições únicas (ano) cada país participou
participacao = todas_part.groupby(['país', 'copa'])['ano'].nunique().reset_index()
participacao.columns = ['país', 'copa', 'num_copas']

# Exibindo Top 5 de cada
for modalidade in ['Feminina', 'Masculina']:
    top5 = participacao[participacao['copa'] == modalidade].nlargest(5, 'num_copas')
    print(top5.to_string(index=False))

#### Q4. 
* Crie um dataframe `gols` com duas colunas `país` e `total_gols` com o total de gols marcados por cada país durante todas as copas do mundo, juntando gols em casa (`gols_1`) e gols como time visitante (`gols_2`).
* Imprima o dataframe `gols` ordenado de forma decrescente, para que os times com mais gols fiquem no topo.

Segue um exemplo ilustrativo com o formato do dataframe resultado:

```
país        total_gols
Brazil      120
Argentina   112
Germany     107
...
```

In [ ]:
gols_casa = wc.groupby('time_1')['gols_1'].sum()
gols_fora = wc.groupby('time_2')['gols_2'].sum()

gols = gols_casa.add(gols_fora, fill_value=0).reset_index()
gols.columns = ['país', 'total_gols']
gols = gols.sort_values(by='total_gols', ascending=False)
display(gols)

#### Q5. 
Qual país tomou mais cartões amarelos somando todas as copas?

Neste exercício você vai trabalhar com as colunas `cartao_amarelo_1` e `cartao_amarelo_2` onde cada observação é uma string represetando uma lista dos cartões amarelos de um único jogo na forma `minuto|placar|jogador(a)`. Por exemplo:
```
['16’|1:0|Rosana Gómez', '20’|2:0|Gabriela Chávez]
```

Recomendo criar colunas `num_cartoes_1` e `num_cartoes_2` a partir de cada coluna `cartao_amarelo_X` usando o método genérico `apply` para chamar uma função que você vai criar para transformar uma observação de cartão amarelo em uma contagem de elementos da lista. 

Lembre de levar em consideração que muitas observações são `NaN`. 

Em seguida faça sua mágica para agrupar as informações por país, acumular os cartões de jogos em casa e jogos visitante, e produzir o resultado final como apresentado a seguir (valores inventados):

```
país        cartões amarelos
Argentina   97
England     93
Australia   93
...
```

In [ ]:
def extrair_qtd_cartoes(obs):
    if pd.isna(obs) or obs == "":
        return 0
    # Como a string representa uma lista, contamos os elementos separados por vírgula
    # ou simplesmente contamos o número de separadores '|' que indicam um registro
    return str(obs).count('|')

wc['num_cartoes_1'] = wc['cartao_amarelo_1'].apply(extrair_qtd_cartoes)
wc['num_cartoes_2'] = wc['cartao_amarelo_2'].apply(extrair_qtd_cartoes)

amarelos_1 = wc.groupby('time_1')['num_cartoes_1'].sum()
amarelos_2 = wc.groupby('time_2')['num_cartoes_2'].sum()

cartoes_final = amarelos_1.add(amarelos_2, fill_value=0).reset_index()
cartoes_final.columns = ['país', 'cartões amarelos']
display(cartoes_final.sort_values(by='cartões amarelos', ascending=False))

#### Q6.
Qual o top10 jogadores com mais gols em copa? Considere gols em jogo e gols de pênalti.

Para conseguir essa informação, você precisará trabalhar com as colunas:
```
10  gols_1_detalhes         
11  gols_2_detalhes    
14  gols_1_penalti     
15  gols_2_penalti     
```

Essas também são colunas textuais, onde cada observações apresenta todos os gols de uma partida separados pelo caracter `|`. Cada gol está na forma `'Jogador(a) · minuto’'`. Por exemplo:
```
'Alex Morgan · 12’|Rose Lavelle · 20’'
```

Lembre de levar em consideração que muitas observações são `NaN`. 

Recomendo criar um dicionário à parte, onde cada chave será um jogador encontrado nessas colunas do dataframe `wc` e o valor correspondente será a contagem de ocorrências desses nomes.

Em seguida basta converter o seu dicionário em um dataframe e imprimí-lo na forma (valores não são inventados 😁):
```
jogador(a)      num_gols 
Marta           17
Miroslav Klose  16
... 
```

In [ ]:
def contabilizar_artilheiros(df):
    dicionario_gols = {}
    colunas_foco = ['gols_1_detalhes', 'gols_2_detalhes', 'gols_1_penalti', 'gols_2_penalti']
    
    for col in colunas_foco:
        for celula in df[col].dropna():
            # Separar múltiplos gols no mesmo jogo (marcados por '|')
            gols_individuais = str(celula).split('|')
            for g in gols_individuais:
                if '·' in g:
                    # O nome vem antes do símbolo '·'
                    nome = g.split('·')[0].strip()
                    dicionario_gols[nome] = dicionario_gols.get(nome, 0) + 1
    return dicionario_gols

artilheiros_dict = contabilizar_artilheiros(wc)
df_artilharia = pd.DataFrame(list(artilheiros_dict.items()), columns=['jogador(a)', 'num_gols'])
display(df_artilharia.sort_values(by='num_gols', ascending=False).head(10))